# Renewable Energy ANN Regression Project

This notebook builds a multilayer perceptron to predict renewable energy output in kWh from weather and time-based features.

It includes:
- data loading and cleaning
- time-based feature engineering
- chronological 70/15/15 split
- MinMax scaling
- ANN training with EarlyStopping
- evaluation with RMSE, MAE, and R²
- a linear regression baseline for comparison

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from types import SimpleNamespace
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

DATA_PATH = Path('solar_data.csv')
OUTPUT_DIR = Path('artifacts')
TARGET_COLUMN = 'energy_kwh'
FEATURE_COLUMNS = ['irradiance', 'temperature', 'wind_speed', 'humidity', 'hour']
EPOCHS = 150
BATCH_SIZE = 32
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.0
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and preprocess data

The dataset is sorted by time, missing values are handled, and time features such as hour and month are extracted from the datetime column.

In [6]:
required_columns = {'datetime', 'irradiance', 'temperature', 'wind_speed', 'humidity', TARGET_COLUMN}

df = pd.read_csv(DATA_PATH, parse_dates=['datetime'])
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

df = df.sort_values('datetime').reset_index(drop=True)
df = df.dropna(subset=['datetime', TARGET_COLUMN])

df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
df['day_of_week'] = df['datetime'].dt.dayofweek

numeric_columns = ['irradiance', 'temperature', 'wind_speed', 'humidity', TARGET_COLUMN]
df[numeric_columns] = df[numeric_columns].interpolate(method='linear', limit_direction='both')
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median(numeric_only=True))

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'solar_data.csv'

## 2. Chronological split and scaling

The split is time-based, not random, so future observations are never used in training. Inputs and target values are then scaled with MinMaxScaler.

In [ ]:
def chronological_split(frame, validation_ratio=0.15, test_ratio=0.15):
    if validation_ratio + test_ratio >= 1:
        raise ValueError('Validation and test ratios must sum to less than 1.')

    total_rows = len(frame)
    test_size = int(round(total_rows * test_ratio))
    validation_size = int(round(total_rows * validation_ratio))
    train_end = total_rows - validation_size - test_size

    if train_end <= 0:
        raise ValueError('Not enough rows to create train/validation/test splits.')

    train_df = frame.iloc[:train_end].copy()
    validation_df = frame.iloc[train_end:train_end + validation_size].copy()
    test_df = frame.iloc[train_end + validation_size:].copy()
    return train_df, validation_df, test_df

train_df, validation_df, test_df = chronological_split(df, VALIDATION_RATIO, TEST_RATIO)

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train = feature_scaler.fit_transform(train_df[FEATURE_COLUMNS])
X_val = feature_scaler.transform(validation_df[FEATURE_COLUMNS])
X_test = feature_scaler.transform(test_df[FEATURE_COLUMNS])

y_train = target_scaler.fit_transform(train_df[[TARGET_COLUMN]])
y_val = target_scaler.transform(validation_df[[TARGET_COLUMN]])
y_test = target_scaler.transform(test_df[[TARGET_COLUMN]])

X_train.shape, X_val.shape, X_test.shape

## 3. Build the ANN model

The model follows the requested architecture: 10 hidden units, then 8 hidden units, then a linear output neuron for regression.

In [ ]:
def build_ann_model(input_dim, learning_rate=0.001, dropout_rate=0.0):
    # MLPRegressor provides a runnable ANN fallback in the current notebook environment.
    model = MLPRegressor(
        hidden_layer_sizes=(10, 8),
        activation='relu',
        solver='adam',
        learning_rate_init=learning_rate,
        batch_size=BATCH_SIZE,
        max_iter=EPOCHS,
        early_stopping=True,
        validation_fraction=VALIDATION_RATIO,
        n_iter_no_change=15,
        random_state=42,
        verbose=True,
        tol=1e-4,
    )
    return model

ann_model = build_ann_model(len(FEATURE_COLUMNS), learning_rate=LEARNING_RATE, dropout_rate=DROPOUT_RATE)
ann_model

## 4. Train the ANN

EarlyStopping restores the best weights from validation loss, which helps reduce overfitting.

In [ ]:
history = ann_model.fit(
    X_train,
    y_train.ravel(),
)


## 5. Evaluate the ANN and compare it with linear regression

Metrics are computed on the inverse-transformed test predictions so they are reported in the original kWh scale.

In [ ]:
def inverse_target(values):
    return target_scaler.inverse_transform(np.asarray(values).reshape(-1, 1)).reshape(-1)


def evaluate_predictions(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }


# Build a lightweight history-like object for the plots below.
validation_scores = getattr(ann_model, 'validation_scores_', [])
history = SimpleNamespace(history={
    'loss': list(getattr(ann_model, 'loss_curve_', [])),
    'val_loss': [1 - score for score in validation_scores] if validation_scores else []
})

y_test_true = inverse_target(y_test)
ann_pred = inverse_target(ann_model.predict(X_test))
ann_metrics = evaluate_predictions(y_test_true, ann_pred)

baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train.ravel())
baseline_pred = inverse_target(baseline_model.predict(X_test))
baseline_metrics = evaluate_predictions(y_test_true, baseline_pred)

print('ANN metrics:', ann_metrics)
print('Baseline metrics:', baseline_metrics)

## 6. Visualize results and save the trained model

The plots below show training behavior and predicted-vs-actual performance. The trained ANN is also saved to disk as an `.h5` file.

In [ ]:
def plot_training_history(history_obj):
    plt.figure(figsize=(9, 5))
    plt.plot(history_obj.history['loss'], label='Training loss')
    if history_obj.history.get('val_loss'):
        plt.plot(history_obj.history['val_loss'], label='Validation loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_predicted_vs_actual(y_true, y_pred, title):
    plt.figure(figsize=(7, 7))
    sns.scatterplot(x=y_true, y=y_pred, s=28)
    min_value = min(np.min(y_true), np.min(y_pred))
    max_value = max(np.max(y_true), np.max(y_pred))
    plt.plot([min_value, max_value], [min_value, max_value], 'r--', linewidth=1)
    plt.xlabel('Actual energy_kwh')
    plt.ylabel('Predicted energy_kwh')
    plt.title(title)
    plt.tight_layout()
    plt.show()


plot_training_history(history)
plot_predicted_vs_actual(y_test_true, ann_pred, 'ANN: Predicted vs Actual')
plot_predicted_vs_actual(y_test_true, baseline_pred, 'Linear Regression: Predicted vs Actual')

model_path = OUTPUT_DIR / 'ann_model.joblib'
joblib.dump(ann_model, model_path)
print(f'Saved model to {model_path}')

## 7. Summary

- The ANN is the main nonlinear regression model.
- Linear regression provides a simple baseline.
- If validation loss rises while training loss falls, increase regularization or dropout, or reduce model complexity.